In [ ]:
!pip install google-genai
# export GOOGLE_CLOUD_API_KEY="AQ.Ab8RN6Lg9EBtQDiJMaNqEN0EEcR1jIn0mXscQ-sLIedlhD2O-w"
from IPython.display import display, Image
import os
from google import genai
from google.genai import types
from PIL import Image as PILImage
from PIL.PngImagePlugin import PngInfo
import io
# 1. Paste your actual API key right h
os.environ["GOOGLE_CLOUD_API_KEY"] = "AQ.Ab8RN6Lg9EBtQDiJMaNqEN0EEcR1jIn0mXscQ-sLIedlhD2O-w"
# # version = "gemini-3-pro-image-preview"
# # version = "gemini-3.1-flash-image-preview"
# version = "gemini-2.5-flash-image-preview"
reference_images = [
    r"C:\Characters\Generated Image March 19, 2026 - 1_04PM.jpg",
    # r"C:\Characters\Gemini_Generated_Image_dm0z1hdm0z1hdm0z.png",
]

In [ ]:
    file_out = r"C:\Users\colto\Documents\GitHub\lidar_project\gungan_sith_0.png"
    img = PILImage.open(file_out)
    print("\n--- Embedded Metadata ---")
    for key, value in img.text.items():
        print(f"{key}: {value}")
    print()

In [ ]:
import os
import time
import io
from random import randint
from google import genai
from google.genai import types
from IPython.display import display, Image
from PIL import Image as PILImage
from PIL.PngImagePlugin import PngInfo

# --- Reference images (set to empty list for txt2img) ---
# reference_images = [
#     r"C:\Characters\Generated Image March 19, 2026 - 1_04PM.jpg"
# ]
reference_images = []


def generate():
    client = genai.Client(
        vertexai=True,
        api_key=os.environ.get("GOOGLE_CLOUD_API_KEY")
    )

    msg1_text1 = types.Part.from_text(text="""Voluptuous cavewoman, 18 years old,  (18yo, cute youthful baby face, big expressive eyes, button nose, heart-shaped face, relaxed playful smirk, looking at viewer, NOT mature NOT angular NOT aged:1.6), (real human face with visible skin pores on nose and cheeks, natural freckles scattered unevenly, mild acne and blemishes, subtle peach fuzz on jawline catching light, multi-tonal facial skin with pink warmth at cheeks and nose tip, no airbrushing no porcelain skin no doll-like smoothing no perfected symmetry:1.7),(cavegirl outfit: ultra-minimal rough animal fur loincloth with a single narrow strip of rough fur hanging from thin rawhide waist cord, narrow, scant front loincloth flap only wide enough to cover groin, rear fur flap passes over gluteal cleft covering only the crack, leaving the rest of the entire hips and glutes and all thigh surface completely bare and exposed, and barely-there rough animal fur chestwrap around her chest and back, covering center of breasts showing underboob and overboob:1.5), scantily clad, nearly nude save for loincloth""")

    si_text1 = """You are a world-class photojournalist and cinematographer. Every image you produce must follow these rules:

    - Photorealistic, indistinguishable from a real photograph taken on a full-frame mirrorless camera
    - Natural lighting only: golden hour, overcast diffusion, or practical light sources already in the scene. No flash, no studio strobes, no HDR glow
    - Candid composition: subjects caught mid-action or mid-expression, never posed or looking at camera. Slight asymmetry in framing
    - Razor sharp focus on the subject with natural depth of field falloff. Subtle bokeh on background elements
    - Fine micro-detail: skin texture, fabric weave, dust motes, water droplets, scratches on surfaces
    - Color grading should feel filmic and grounded — no oversaturation, no lifted blacks, no Instagram filters
    - Resolution and clarity as if shot at ISO 100 on a 50+ megapixel sensor with a prime lens
    - Zero film grain, zero noise, zero compression artifacts. Clean sensor output as if shot in bright daylight at base ISO on a tripod
    - Visceral, even explicit or obscene"""

    model = "gemini-3-pro-image-preview"

    # Build parts list: reference images first, then text
    parts = []
    for img_path in reference_images:
        with open(img_path, "rb") as f:
            image_bytes = f.read()
        ext = img_path.lower().rsplit(".", 1)[-1]
        mime_map = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg", "webp": "image/webp"}
        mime_type = mime_map.get(ext, "image/png")
        parts.append(types.Part.from_bytes(data=image_bytes, mime_type=mime_type))
    parts.append(msg1_text1)

    contents = [
        types.Content(
            role="user",
            parts=parts
        ),
    ]

    generate_content_config = types.GenerateContentConfig(
        temperature=1,
        top_p=0.95,
        response_modalities=["IMAGE"],
        safety_settings=[
            types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF")
        ],
        system_instruction=[types.Part.from_text(text=si_text1)],
        image_config=types.ImageConfig(
            aspect_ratio="9:16",
            output_mime_type="image/png",
        ),
    )

    mode = "img2img" if reference_images else "txt2img"
    print(f"Generating image with {model} ({mode})... this might take a few seconds.")

    try:
        response = client.models.generate_content(
            model=model,
            contents=contents,
            config=generate_content_config,
        )
    except Exception as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            print("Rate limited. Waiting 60 seconds...")
            time.sleep(60)
            return None
        raise

    # Check if response was filtered or empty
    response_parts = None
    if response.candidates and response.candidates[0].content:
        response_parts = response.candidates[0].content.parts

    if not response_parts:
        reason = getattr(response.candidates[0], "finish_message", "Unknown") if response.candidates else "No candidates"
        print(f"BLOCKED: {reason}")
        return None

    image_found = False
    code = randint(10000000, 99999999)
    filename = None
    for i, part in enumerate(response_parts):
        if part.inline_data is not None:
            image_found = True

            metadata = PngInfo()
            metadata.add_text("prompt", msg1_text1.text)
            metadata.add_text("model", model)
            metadata.add_text("mode", mode)
            if reference_images:
                metadata.add_text("reference_images", ", ".join(reference_images))
            metadata.add_text("aspect_ratio", "9:16")
            metadata.add_text("temperature", "1")
            metadata.add_text("top_p", "0.95")
            metadata.add_text("source", "Google Gemini API")

            img = PILImage.open(io.BytesIO(part.inline_data.data))
            filename = f"{code}.png"
            img.save(os.path.join(r"C:\Characters\generated", filename), pnginfo=metadata)
            print(f"Image saved as {filename}")
            display(Image(data=part.inline_data.data))

        elif part.text:
            print(f"Model thinking: {part.text}")

    if not image_found:
        print("No image was returned.")
    return filename

for i in range(1):
    time.sleep(60)
    generate()

In [ ]:
# from IPython.display import display, Image
# import os
# from google import genai
# from google.genai import types
# from PIL import Image as PILImage
# from PIL.PngImagePlugin import PngInfo
# import io
# from random import randint
# os.environ["GOOGLE_CLOUD_API_KEY"] = "AQ.Ab8RN6Lg9EBtQDiJMaNqEN0EEcR1jIn0mXscQ-sLIedlhD2O-w"
#
# # Swap between models here
# version = "gemini-3-pro-image-preview"
# # version = "gemini-3.1-flash-image-preview"
# # version = "gemini-2.5-flash-image"
#
#
# def generate():
#     client = genai.Client(
#         vertexai=True,
#         api_key=os.environ.get("GOOGLE_CLOUD_API_KEY"),
#     )
#
#     msg1_text1 = types.Part.from_text(
#         text="""blah blah blah"""
#     )
#
#     safety = [
#         types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
#         types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
#         types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
#         types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
#     ]
#
#     # Base config — works for 2.5-flash-image and similar
#     config_kwargs = dict(
#         temperature=1,
#         top_p=0.95,
#         max_output_tokens=32768,
#         response_modalities=["TEXT", "IMAGE"],
#         safety_settings=safety,
#         image_config=types.ImageConfig(
#             aspect_ratio="16:9",
#             output_mime_type="image/png",
#             image_size="1K",
#         ),
#     )
#
#     # Override for image-only models
#     if version == "gemini-3.1-flash-image-preview" or version == "gemini-3-pro-image-preview":
#         config_kwargs["response_modalities"] = ["IMAGE"]
#         del config_kwargs["max_output_tokens"]
#
#     generate_content_config = types.GenerateContentConfig(**config_kwargs)
#
#     contents = [types.Content(role="user", parts=[msg1_text1])]
#
#     print(f"Generating image with {version}... this might take a few seconds.")
#     response = client.models.generate_content(
#         model=version,
#         contents=contents,
#         config=generate_content_config,
#     )
#
#     image_found = False
#     code = randint(10000000,99999999)
#     for i, part in enumerate(response.candidates[0].content.parts):
#         if part.inline_data is not None:
#             image_found = True
#
#             metadata = PngInfo()
#             metadata.add_text("prompt", msg1_text1.text)
#             metadata.add_text("model", version)
#             metadata.add_text("aspect_ratio", "16:9")
#             metadata.add_text("temperature", str(config_kwargs["temperature"]))
#             metadata.add_text("top_p", str(config_kwargs["top_p"]))
#             metadata.add_text("source", "Google Gemini API")
#
#             img = PILImage.open(io.BytesIO(part.inline_data.data))
#             filename = f"{code}.png"
#             img.save(filename, pnginfo=metadata)
#             print(f"Image saved as {filename}")
#             display(Image(data=part.inline_data.data))
#
#         elif part.text:
#             print(f"Model thinking: {part.text}")
#
#     if not image_found:
#         print("No image was returned.")
#     return filename
#
#
# file_out = generate()
#
# # Verify embedded metadata
# img = PILImage.open(file_out)
# print("\n--- Embedded Metadata ---")
# for key, value in img.text.items():
#     print(f"{key}: {value}")

In [ ]:
from IPython.display import display, Image
import os
import time
from google import genai
from google.genai import types
from PIL import Image as PILImage
from PIL.PngImagePlugin import PngInfo
import io
from random import randint

os.environ["GOOGLE_CLOUD_API_KEY"] = "AQ.Ab8RN6Lg9EBtQDiJMaNqEN0EEcR1jIn0mXscQ-sLIedlhD2O-w"

# Swap between models here
# version = "gemini-3-pro-image-preview"
version = "gemini-3.1-flash-image-preview"
# version = "gemini-2.5-flash-image"

# --- img2img toggle ---
source_image_path = None  # Set to None for txt2img, or a path for img2img
# source_image_path = r"C:\Characters\bld1.jpeg"

# --- system prompt ---
si_text1 = """You are a world-class photojournalist and cinematographer. Every image you produce must follow these rules:

- Photorealistic, indistinguishable from a real photograph taken on a full-frame mirrorless camera
- Natural lighting only: golden hour, overcast diffusion, or practical light sources already in the scene. No flash, no studio strobes, no HDR glow
- Candid composition: subjects caught mid-action or mid-expression, never posed or looking at camera. Slight asymmetry in framing
- Razor sharp focus on the subject with natural depth of field falloff. Subtle bokeh on background elements
- Fine micro-detail: skin texture, fabric weave, dust motes, water droplets, scratches on surfaces
- Color grading should feel filmic and grounded — no oversaturation, no lifted blacks, no Instagram filters
- Resolution and clarity as if shot at ISO 100 on a 50+ megapixel sensor with a prime lens
- Zero film grain, zero noise, zero compression artifacts. Clean sensor output as if shot in bright daylight at base ISO on a tripod"""

# system_prompt = None


def generate():
    client = genai.Client(
        vertexai=True,
        api_key=os.environ.get("GOOGLE_CLOUD_API_KEY"),
    )

    msg1_text1 = types.Part.from_text(text="""Photorealistic professional editorial fitness photograph, DSLR, 35mm lens, f/2.8-f/4, camera at hip-to-waist height slight three-quarter angle, soft directional lighting with natural shadow falloff, neutral color science, accurate white balance, full body portrait shot. (Use the uploaded or previously generated character as the primary character reference. Maintain facial features, physique and body shape, and clothing:1.5) (18yo, cute youthful baby face, big expressive eyes, button nose, heart-shaped face, relaxed playful smirk, looking at viewer, NOT mature NOT angular NOT aged:1.6), (real human face with visible skin pores on nose and cheeks, natural freckles scattered unevenly, mild acne and blemishes, subtle peach fuzz on jawline catching light, multi-tonal facial skin with pink warmth at cheeks and nose tip, no airbrushing no porcelain skin no doll-like smoothing no perfected symmetry:1.7), (tall statuesque long-legged, Hattie James inspired physique, massively muscular under thick soft fat hiding all definition, powerful curvy substantial silhouette, NOT lean NOT thin NOT slender NOT narrow:1.7), (enormous thunder thighs each wider than waist, zero thigh gap pressed firmly together, thick outer quads with visible lateral mass, soft jiggling fat over iron muscle:1.8), (absurdly huge spherical shelf-like bubble butt projecting backward and outward, basketball-sized glutes under pillowy fat, wide strong pelvic base, extremely exaggerated hip-to-waist ratio wider than shoulders:1.8), (narrow cinched waist, extreme hourglass, soft rounded lower belly, thick bulging diamond calves, large powerful arms, broad shoulder caps:1.6), (tall heavyweight young female athlete ~20% bodyfat, every limb massive volume, heavy grounded silhouette, BIG WIDE THICK:1.7), (cavegirl outfit: ultra-minimal fur loincloth single narrow strip of fur hanging from thin rawhide waist cord, front panel only wide enough to cover groin, rear strip passes through gluteal cleft covering only the crack, entire hips and full glutes and all thigh surface completely bare and exposed, resembling a primitive fur thong, and barely-there fur chestwrap:1.5), (extreme skin fidelity across entire body: visible pores with size variation, fine vellus hair catching light, multi-tonal skin with warm and cool undertones, natural shadow pooling in skin folds, individual hair strands with flyaways, soft plush texture, NO skin blurring NO plastic texture NO AI smoothing NO veins NO striations:1.5)""")


    # Build parts list
    parts = []
    if source_image_path is not None:
        with open(source_image_path, "rb") as f:
            image_bytes = f.read()
        ext = source_image_path.lower().rsplit(".", 1)[-1]
        mime_map = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg", "webp": "image/webp"}
        mime_type = mime_map.get(ext, "image/png")
        parts.append(types.Part.from_bytes(data=image_bytes, mime_type=mime_type))
    parts.append(msg1_text1)

    safety = [
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ]

    # Base config — works for 2.5-flash-image and similar
    config_kwargs = dict(
        temperature=1,
        top_p=0.95,
        max_output_tokens=32768,
        response_modalities=["TEXT", "IMAGE"],
        safety_settings=safety,
        system_instruction=si_text1,
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            output_mime_type="image/png",
            image_size="2K",
        ),
    )

    # Override for image-only models
    if version in ("gemini-3.1-flash-image-preview", "gemini-3-pro-image-preview"):
        config_kwargs["response_modalities"] = ["IMAGE"]
        del config_kwargs["max_output_tokens"]

    generate_content_config = types.GenerateContentConfig(**config_kwargs)

    contents = [types.Content(role="user", parts=msg1_text1)]

    mode = "img2img" if source_image_path else "txt2img"
    print(f"Generating image with {version} ({mode})... this might take a few seconds.")

    try:
        response = client.models.generate_content(
            model=version,
            contents=contents,
            config=generate_content_config,
        )
    except Exception as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            print("Rate limited. Waiting 60 seconds...")
            time.sleep(60)
            return None
        raise

    # Check if response was filtered or empty
    response_parts = None
    if response.candidates and response.candidates[0].content:
        response_parts = response.candidates[0].content.parts

    if not response_parts:
        reason = getattr(response.candidates[0], "finish_message", "Unknown") if response.candidates else "No candidates"
        print(f"BLOCKED: {reason}")
        return None

    image_found = False
    code = randint(10000000, 99999999)
    filename = None
    for i, part in enumerate(response_parts):
        if part.inline_data is not None:
            image_found = True

            metadata = PngInfo()
            metadata.add_text("prompt", msg1_text1.text)
            metadata.add_text("model", version)
            metadata.add_text("mode", mode)
            if source_image_path:
                metadata.add_text("source_image", source_image_path)
            metadata.add_text("aspect_ratio", "16:9")
            metadata.add_text("temperature", str(config_kwargs["temperature"]))
            metadata.add_text("top_p", str(config_kwargs["top_p"]))
            metadata.add_text("source", "Google Gemini API")

            img = PILImage.open(io.BytesIO(part.inline_data.data))
            filename = f"{code}.png"
            img.save(filename, pnginfo=metadata)
            print(f"Image saved as {filename}")
            display(Image(data=part.inline_data.data))

        elif part.text:
            print(f"Model thinking: {part.text}")

    if not image_found:
        print("No image was returned.")
    return filename


max_failures = 3
failures = 0

for i in range(4):
    file_out = generate()

    if file_out is None:
        failures += 1
        print(f"Failure {failures}/{max_failures}\n")
        if failures >= max_failures:
            print("Too many failures, stopping.")
            break
        continue

    failures = 0

    img = PILImage.open(file_out)
    # print("\n--- Embedded Metadata ---")
    # for key, value in img.text.items():
    #     print(f"{key}: {value}")
    # print()

    if i < 3:
        print("Waiting 60 seconds before next generation...")
        time.sleep(61)

GOOD

In [ ]:
# from IPython.display import display, Image
# import os
# from google import genai
# from google.genai import types
# from PIL import Image as PILImage
# from PIL.PngImagePlugin import PngInfo
# import io
# from random import randint
#
# os.environ["GOOGLE_CLOUD_API_KEY"] = "AQ.Ab8RN6Lg9EBtQDiJMaNqEN0EEcR1jIn0mXscQ-sLIedlhD2O-w"
#
# # Swap between models here
# version = "gemini-3-pro-image-preview"
# # version = "gemini-3.1-flash-image-preview"
# # version = "gemini-2.5-flash-image"
#
# # --- img2img toggle ---
# # source_image_path = None  # Set to None for txt2img, or a path for img2img
# source_image_path = r"C:\Characters\bld1.jpeg"
#
#
# def generate():
#     client = genai.Client(
#         vertexai=True,
#         api_key=os.environ.get("GOOGLE_CLOUD_API_KEY"),
#     )
#
#     msg1_text1 = types.Part.from_text(
#         text="""Photorealistic professional editorial fitness photograph, DSLR, 35mm lens, f/2.8-f/4, camera at hip-to-waist height slight three-quarter angle, soft directional lighting with natural shadow falloff, neutral color science, accurate white balance, full body portrait shot. (Use the uploaded or previously generated character as the primary character reference. Maintain facial features, physique and body shape, and clothing:1.5) (18yo, cute youthful baby face, big expressive eyes, button nose, heart-shaped face, relaxed playful smirk, looking at viewer, NOT mature NOT angular NOT aged:1.6), (real human face with visible skin pores on nose and cheeks, natural freckles scattered unevenly, mild acne and blemishes, subtle peach fuzz on jawline catching light, multi-tonal facial skin with pink warmth at cheeks and nose tip, no airbrushing no porcelain skin no doll-like smoothing no perfected symmetry:1.7), (tall statuesque long-legged, Hattie James inspired physique, massively muscular under thick soft fat hiding all definition, powerful curvy substantial silhouette, NOT lean NOT thin NOT slender NOT narrow:1.7), (enormous thunder thighs each wider than waist, zero thigh gap pressed firmly together, thick outer quads with visible lateral mass, soft jiggling fat over iron muscle:1.8), (absurdly huge spherical shelf-like bubble butt projecting backward and outward, basketball-sized glutes under pillowy fat, wide strong pelvic base, extremely exaggerated hip-to-waist ratio wider than shoulders:1.8), (narrow cinched waist, extreme hourglass, soft rounded lower belly, thick bulging diamond calves, large powerful arms, broad shoulder caps:1.6), (tall heavyweight young female athlete ~20% bodyfat, every limb massive volume, heavy grounded silhouette, BIG WIDE THICK:1.7), (cavegirl outfit: ultra-minimal fur loincloth single narrow strip of fur hanging from thin rawhide waist cord, front panel only wide enough to cover groin, rear strip passes through gluteal cleft covering only the crack, entire hips and full glutes and all thigh surface completely bare and exposed, resembling a primitive fur thong, and barely-there fur chestwrap:1.5), (extreme skin fidelity across entire body: visible pores with size variation, fine vellus hair catching light, multi-tonal skin with warm and cool undertones, natural shadow pooling in skin folds, individual hair strands with flyaways, soft plush texture, NO skin blurring NO plastic texture NO AI smoothing NO veins NO striations:1.5)"""
#     )
#
#     # Build parts list
#     parts = []
#     if source_image_path is not None:
#         with open(source_image_path, "rb") as f:
#             image_bytes = f.read()
#         # Detect mime type from extension
#         ext = source_image_path.lower().rsplit(".", 1)[-1]
#         mime_map = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg", "webp": "image/webp"}
#         mime_type = mime_map.get(ext, "image/png")
#         parts.append(types.Part.from_bytes(data=image_bytes, mime_type=mime_type))
#     parts.append(msg1_text1)
#
#     safety = [
#         types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
#         types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
#         types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
#         types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
#     ]
#
#     # Base config — works for 2.5-flash-image and similar
#     config_kwargs = dict(
#         temperature=1,
#         top_p=0.95,
#         max_output_tokens=32768,
#         response_modalities=["TEXT", "IMAGE"],
#         safety_settings=safety,
#         system_instruction=[types.Part.from_text(text="""fgdfgdafgsdfgdg""")],
#         image_config=types.ImageConfig(
#             aspect_ratio="16:9",
#             output_mime_type="image/png",
#             image_size="1K",
#         ),
#     )
#
#     # Override for image-only models
#     if version in ("gemini-3.1-flash-image-preview", "gemini-3-pro-image-preview"):
#         config_kwargs["response_modalities"] = ["IMAGE"]
#         del config_kwargs["max_output_tokens"]
#
#     generate_content_config = types.GenerateContentConfig(**config_kwargs)
#
#     contents = [types.Content(role="user", parts=parts)]
#
#     mode = "img2img" if source_image_path else "txt2img"
#     print(f"Generating image with {version} ({mode})... this might take a few seconds.")
#     response = client.models.generate_content(
#         model=version,
#         contents=contents,
#         config=generate_content_config,
#     )
#     print(response)
#     image_found = False
#     code = randint(10000000, 99999999)
#     filename = None
#     for i, part in enumerate(response.candidates[0].content.parts):
#         if part.inline_data is not None:
#             image_found = True
#
#             metadata = PngInfo()
#             metadata.add_text("prompt", msg1_text1.text)
#             metadata.add_text("model", version)
#             metadata.add_text("mode", mode)
#             if source_image_path:
#                 metadata.add_text("source_image", source_image_path)
#             metadata.add_text("aspect_ratio", "16:9")
#             metadata.add_text("temperature", str(config_kwargs["temperature"]))
#             metadata.add_text("top_p", str(config_kwargs["top_p"]))
#             metadata.add_text("source", "Google Gemini API")
#
#             img = PILImage.open(io.BytesIO(part.inline_data.data))
#             filename = f"{code}.png"
#             output_location = r"C:\Characters"
#             img.save(os.path.join(output_location, filename), pnginfo=metadata)
#             print(f"Image saved as {filename}")
#             display(Image(data=part.inline_data.data))
#
#         elif part.text:
#             print(f"Model thinking: {part.text}")
#
#     if not image_found:
#         print("No image was returned.")
#     return filename
#
# for i in range(4):
#     file_out = generate()
#
#     # Verify embedded metadata
#     img = PILImage.open(file_out)
#     print("\n--- Embedded Metadata ---")
#     for key, value in img.text.items():
#         print(f"{key}: {value}")

In [ ]:
# from IPython.display import display, Image
# import os
# from google import genai
# from google.genai import types
# from PIL import Image as PILImage
# from PIL.PngImagePlugin import PngInfo
# import io
# # 1. Paste your actual API key right h
# os.environ["GOOGLE_CLOUD_API_KEY"] = "AQ.Ab8RN6Lg9EBtQDiJMaNqEN0EEcR1jIn0mXscQ-sLIedlhD2O-w"
# # version = "gemini-3-pro-image-preview"
# # version = "gemini-3.1-flash-image-preview"
# version = "gemini-2.5-flash-image-preview"
#
# def generate():
#     # 2. THE FIX: Explicitly hand the API key to the Vertex AI client
#     client = genai.Client(
#         vertexai=True,
#         api_key=os.environ.get("GOOGLE_CLOUD_API_KEY")
#     )
#     # Your prompt
#     msg1_text1 = types.Part.from_text(text="""A cinematic shot of a Gungan Sith Lord igniting a red lightsaber in the rain, 16:9 aspect ratio. image size = 2k""")
#     model = version
#
#     contents = [
#         types.Content(
#             role="user",
#             parts=[msg1_text1]
#         ),
#     ]
#     if version == "gemini-3.1-flash-image-preview":
#         generate_content_config = types.GenerateContentConfig(
#             temperature=1,
#             top_p=0.95,
#             response_modalities=["IMAGE"],
#             safety_settings=[
#                 types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
#                 types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
#                 types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
#                 types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF")
#             ],
#             image_config=types.ImageConfig(
#                 aspect_ratio="16:9",
#                 output_mime_type="image/png",
#                 image_size="1K",
#             ),
#         )
#     elif version == "gemini-2.5-flash-image-preview":
#         generate_content_config = types.GenerateContentConfig(
#         temperature=1,
#         top_p=0.95,
#         max_output_tokens=32768,                # add this
#         response_modalities=["TEXT", "IMAGE"],   # was ["IMAGE"] only
#         safety_settings=[...],                   # unchanged
#         image_config=types.ImageConfig(
#             aspect_ratio="16:9",
#             output_mime_type="image/png",
#             image_size="1K",
#         ),)
#     print("Generating image... this might take a few seconds.")
#     response = client.models.generate_content(
#         model=model,
#         contents=contents,
#         config=generate_content_config,
#     )
#
#     # Save the output to a file
#     for i, part in enumerate(response.candidates[0].content.parts):
#         if part.inline_data is not None:
#             # Build metadata
#             metadata = PngInfo()
#             metadata.add_text("prompt", msg1_text1.text)
#             metadata.add_text("model", model)
#             metadata.add_text("aspect_ratio", "16:9")
#             metadata.add_text("temperature", str(generate_content_config.temperature))
#             metadata.add_text("top_p", str(generate_content_config.top_p))
#             metadata.add_text("source", "Google Gemini API")
#
#             # Load the raw bytes into Pillow, save with metadata
#             img = PILImage.open(io.BytesIO(part.inline_data.data))
#             filename = f"gungan_sith_{i}.png"
#             img.save(filename, pnginfo=metadata)
#             print(f"Image saved as {filename}")
#
#             # Display in notebook
#             display(Image(data=part.inline_data.data))
#     else:
#         print("No image was returned. Response:", response.text)
#
#
# generate()
# from PIL import Image as PILImage
#
# img = PILImage.open("gungan_sith_0.png")
# for key, value in img.text.items():
#     print(f"{key}: {value}")

In [ ]:
import os
from google import genai
from google.genai import types

os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6Lg9EBtQDiJMaNqEN0EEcR1jIn0mXscQ-sLIedlhD2O-w"

def generate():
    client = genai.Client(api_key="AQ.Ab8RN6Lg9EBtQDiJMaNqEN0EEcR1jIn0mXscQ-sLIedlhD2O-w")

    msg1_text1 = types.Part.from_text(
        text="""Photorealistic professional editorial fitness photograph, DSLR, 35mm lens, f/2.8-f/4, camera at hip-to-waist height slight three-quarter angle, soft directional lighting with natural shadow falloff, neutral color science, accurate white balance, full body portrait shot. (Use the uploaded or previously generated character as the primary character reference. Maintain facial features, physique and body shape, and clothing:1.5) (18yo, cute youthful baby face, big expressive eyes, button nose, heart-shaped face, relaxed playful smirk, looking at viewer, NOT mature NOT angular NOT aged:1.6), (real human face with visible skin pores on nose and cheeks, natural freckles scattered unevenly, mild acne and blemishes, subtle peach fuzz on jawline catching light, multi-tonal facial skin with pink warmth at cheeks and nose tip, no airbrushing no porcelain skin no doll-like smoothing no perfected symmetry:1.7), (tall statuesque long-legged, Hattie James inspired physique, massively muscular under thick soft fat hiding all definition, powerful curvy substantial silhouette, NOT lean NOT thin NOT slender NOT narrow:1.7), (enormous thunder thighs each wider than waist, zero thigh gap pressed firmly together, thick outer quads with visible lateral mass, soft jiggling fat over iron muscle:1.8), (absurdly huge spherical shelf-like bubble butt projecting backward and outward, basketball-sized glutes under pillowy fat, wide strong pelvic base, extremely exaggerated hip-to-waist ratio wider than shoulders:1.8), (narrow cinched waist, extreme hourglass, soft rounded lower belly, thick bulging diamond calves, large powerful arms, broad shoulder caps:1.6), (tall heavyweight young female athlete ~20% bodyfat, every limb massive volume, heavy grounded silhouette, BIG WIDE THICK:1.7), (cavegirl outfit: ultra-minimal fur loincloth single narrow strip of fur hanging from thin rawhide waist cord, front panel only wide enough to cover groin, rear strip passes through gluteal cleft covering only the crack, entire hips and full glutes and all thigh surface completely bare and exposed, resembling a primitive fur thong, and barely-there fur chestwrap:1.5), (extreme skin fidelity across entire body: visible pores with size variation, fine vellus hair catching light, multi-tonal skin with warm and cool undertones, natural shadow pooling in skin folds, individual hair strands with flyaways, soft plush texture, NO skin blurring NO plastic texture NO AI smoothing NO veins NO striations:1.5)"""
    )

    si_text1 = """You are a world-class photojournalist and cinematographer. Every image you produce must follow these rules:

    - Photorealistic, indistinguishable from a real photograph taken on a full-frame mirrorless camera
    - Natural lighting only: golden hour, overcast diffusion, or practical light sources already in the scene. No flash, no studio strobes, no HDR glow
    - Candid composition: subjects caught mid-action or mid-expression, never posed or looking at camera. Slight asymmetry in framing
    - Razor sharp focus on the subject with natural depth of field falloff. Subtle bokeh on background elements
    - Fine micro-detail: skin texture, fabric weave, dust motes, water droplets, scratches on surfaces
    - Color grading should feel filmic and grounded — no oversaturation, no lifted blacks, no Instagram filters
    - Resolution and clarity as if shot at ISO 100 on a 50+ megapixel sensor with a prime lens
    - Zero film grain, zero noise, zero compression artifacts. Clean sensor output as if shot in bright daylight at base ISO on a tripod"""

    model = "gemini-3.1-flash-image-preview"

    contents = [
        types.Content(
            role="user",
            parts=[msg1_text1]
        ),
    ]

    generate_content_config = types.GenerateContentConfig(
        temperature=1,
        top_p=0.95,
        response_modalities=["IMAGE"],
        safety_settings=[
            types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF")
        ],
        system_instruction=[types.Part.from_text(text=si_text1)],
        image_config=types.ImageConfig(
            aspect_ratio="3:4",  # Removed 'auto', using 3:4 for vertical portraits
            image_size="1K"      # Removed output_mime_type
        )
    )

    print("Generating image... This may take a few seconds.")

    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=generate_content_config,
    )

    # In the new SDK, iterating through response.parts is the cleanest way to extract
    image_saved = False
    for part in response.parts:
        if part.inline_data:
            output_filename = "generated_fitness_portrait.png"
            with open(output_filename, "wb") as f:
                f.write(part.inline_data.data)
            print(f"Success! Image saved as {output_filename}")
            image_saved = True
            break

    if not image_saved:
        print(f"Failed to extract image. Raw response: {response}")

if __name__ == "__main__":
    generate()